In [ ]:
from copy import deepcopy
from re import findall

import matplotlib.pyplot as plt
import pandas as pd
from nn import *
from ray import train, tune
from scipy.optimize import bisect


def arg(**_):
    o = {}
    for i, _ in _.items():
        for i in i.split("_"):
            t = o
            i = findall("([A-Za-z][a-z]*)([0-9]*)", i)
            for j, k in i[:-1]:
                j = j.lower()
                if k:
                    if j not in t:
                        t[j] = [{}]
                    t[j] += [deepcopy(t[j][-1]) for k in range(int(k) + 1 - len(t[j]))]
                    t = t[j][int(k)]
                else:
                    if j not in t:
                        t[j] = {}
                    t = t[j]
            t[i[-1][0]] = _
    return o


class param(dict):
    def __mul__(s, x):
        return param({_: (s * x).__ceil__() if isinstance(s, float) else s for _, s in s.items()})


def decay(x, k, d=-4, shift=1, logspace=False):
    if shift:
        x = x[(..., slice(-shift), *[slice(None)] * (-d - 1))]
    k = k * torch.arange(x.size(d) / 2, -x.size(d) / 2, -1, dtype=x.dtype, device=x.device)[(..., *[None] * (-d - 1))]
    if logspace:
        x = x.sign().mul_(x.abs_().add_(5e-41).log_().add_(k).logcumsumexp(d).sub_(a.logcumsumexp(k)).exp_())
    else:
        x = x.mul_(k.exp_()).cumsum_(d).div_(k.cumsum(d))
    if shift:
        x = functional.pad(x, [*[0, 0] * (-d - 1), shift, 0])
    return x


def Ret(i, h, k, v, o, r=1):
    return Seq(
        Dot(
            Seq(
                Seq(Linear(i, h * k), Unflatten(-1, [h, 1, k])),
            ),
            Seq(
                Tup(
                    Mul(
                        Seq(Linear(i, h * k), Unflatten(-1, [h, 1, k])),
                        Seq(Linear(i, h * v, False), Unflatten(-1, [h, v, 1])),
                    ),
                    Add(
                        Const(Parameter(init.uniform_(torch.empty(h, 1, k), -0.5, -0.2))),
                        Const(Parameter(init.uniform_(torch.empty(h, v, 1), -0.5, -0.2))),
                    ),
                ),
                F(decay, shift=r, unpak=True),
            ),
        ),
        Flatten(-3),
        Linear(h * v, o, False),
    )


def Cov(i, h, o=0, a=0, r=1):
    return Seq(
        SepConvLeft(i, h, o or a, r),
        not o and Seq(BatchNorm(a), SiLU(True)),
    )


def Blk(i, h, o, r=[], c=[], b=False):
    return Add(
        Seq(
            Add(
                Seq(*[Ret(**r) for r in r]),
                Seq(*[Cov(**c) for c in c]),
            ),
            BatchNorm(h),
            SiLU(True),
            Linear(h, o, False),
        ),
        Linear(i, o, b),
    )


def Blks(e, i, b=[]):
    return Seq(
        Embedding(e, i),
        *[Blk(**b) for b in b],
    )


def get_blks(_):
    sz = _.pop("sz", 25000)
    _ = param(_)
    return Blks(**arg(**_ * bisect(lambda x: sum(_.numel() for _ in Blks(**arg(**_ * x)).parameters()) - sz, 4, 64, xtol=0.1)))


def train_blks(
    _,
    a=("py.txt", 32, 256),
    m=("cuda", torch.bfloat16),
    O={"max_lr": 0.005, "total_steps": 10000, "pct_start": 0.01, "div_factor": 9, "final_div_factor": 99},
    report=train.report,
    tqdm=lambda x: x,
    trace=torch.jit.trace,
):
    a = Dat(*a)
    m = get_blks(_).to(*m)
    o = torch.optim.Adam(m.parameters())
    O = torch.optim.lr_scheduler.OneCycleLR(o, **O)
    X = a()
    m = trace(m, X[:, :-1])
    L = A = 0
    for _ in tqdm(range(1, O.total_steps + 1)):
        x, X = X, a()
        y = m(x[:, :-1])
        x = x[:, 1:]
        l = functional.cross_entropy(y.mT, x)
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
        L += l.detach()
        A += (y.argmax(2) == x).sum()
        if not _ % 100:
            report({"l": float(L), "a": float(A) / x.numel()})

In [ ]:
tuner = tune.Tuner(
    tune.with_resources(train_blks, {"gpu": 0.5}),
    param_space=dict(
        e_b1o=68,
        i_b0i_b0r0i_b0c0i=1.0,
        b0r0h=tune.loguniform(0.2, 1),
        b0r0k=tune.loguniform(0.2, 1),
        b0r0v=tune.loguniform(0.2, 1),
        b0c0h=tune.loguniform(0.2, 1),
        b0c0a_b0c1i=tune.loguniform(0.2, 1),
        b0c1h=tune.loguniform(0.2, 1),
        b0r0o_b0c1o_b0h=tune.loguniform(1, 4),
        b0o_b1i_b1r0i_b1c0i=tune.loguniform(1, 4),
        b1r0h=tune.loguniform(0.2, 1),
        b1r0k=tune.loguniform(0.2, 1),
        b1r0v=tune.loguniform(0.2, 1),
        b1c0h=tune.loguniform(0.2, 1),
        b1c0a_b1c1i=tune.loguniform(0.2, 1),
        b1c1h=tune.loguniform(0.2, 1),
        b1r0o_b1c1o_b1h=tune.loguniform(1, 4),
        b0r0r=tune.randint(0, 6),
        b0c0r=tune.randint(1, 6),
        b0c1r=tune.randint(1, 6),
        b1r0r=tune.randint(0, 6),
        b1c0r=tune.randint(1, 6),
        b1c1r=tune.randint(1, 6),
        sz=100000,
    ),
    tune_config=tune.TuneConfig(
        mode="min",
        metric="l",
        num_samples=1000,
        max_concurrent_trials=2,
        scheduler=tune.schedulers.ASHAScheduler(max_t=100, grace_period=3),
    ),
    run_config=train.RunConfig(name="a6"),
)
a6 = tuner.fit()

In [ ]:
df = a6.get_dataframe()
df = df[df.training_iteration > 99][["l", "a"] + [i for i in df.columns if "/" in i and df[i].std() > 0]].rename(columns=lambda x: x.split("/")[-1]).sort_values("l")
df

In [ ]:
pd.DataFrame({"min": df.min(), "median": df.median(), "mean": df.mean(), "max": df.max(), "std": df.std()})

In [ ]:
for i in a6:
    plt.plot(i.metrics_dataframe.a)

In [ ]:
tuner = tune.Tuner(
    tune.with_resources(train_blks, {"gpu": 0.5}),
    param_space=dict(
        e_b1o=68,
        i_b0i_b0r0i_b0c0i=1.0,
        b0r0h=tune.loguniform(0.2, 1),
        b0r0k=tune.loguniform(0.2, 1),
        b0r0v=tune.loguniform(0.2, 1),
        b0c0h=tune.loguniform(0.2, 1),
        b0c0a_b0c1i=tune.loguniform(0.2, 1),
        b0c1h=tune.loguniform(0.2, 1),
        b0r0o_b0c1o_b0h=tune.loguniform(1, 4),
        b0o_b1i_b1r0i_b1c0i=tune.loguniform(1, 4),
        b1r0h=tune.loguniform(0.2, 1),
        b1r0k=tune.loguniform(0.2, 1),
        b1r0v=tune.loguniform(0.2, 1),
        b1c0h=tune.loguniform(0.2, 1),
        b1c0a_b1c1i=tune.loguniform(0.2, 1),
        b1c1h=tune.loguniform(0.2, 1),
        b1r0o_b1c1o_b1h=tune.loguniform(1, 4),
        b0r0r=tune.randint(0, 6),
        b0c0r=tune.randint(1, 6),
        b0c1r=tune.randint(1, 6),
        b1r0r=tune.randint(0, 6),
        b1c0r=tune.randint(1, 6),
        b1c1r=tune.randint(1, 6),
        sz=1000000,
    ),
    tune_config=tune.TuneConfig(
        mode="min",
        metric="l",
        num_samples=500,
        max_concurrent_trials=2,
        scheduler=tune.schedulers.ASHAScheduler(max_t=100, grace_period=3),
    ),
    run_config=train.RunConfig(name="a7"),
)
a7 = tuner.fit()

In [ ]:
df = a7.get_dataframe()
df = df[df.training_iteration > 99][["l", "a"] + [i for i in df.columns if "/" in i and df[i].std() > 0]].rename(columns=lambda x: x.split("/")[-1]).sort_values("l")
df

In [ ]:
pd.DataFrame({"min": df.min(), "median": df.median(), "mean": df.mean(), "max": df.max(), "std": df.std()})

In [ ]:
for i in a7:
    plt.plot(i.metrics_dataframe.a)